# S04 - N-Gram Language Models
## Solutions

### Exercise 1 (Easy)

In [ ]:
sentence = "I love natural language processing"

def get_ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

tokens = sentence.lower().split()
print("Bigrams:", get_ngrams(tokens, 2))
print("Trigrams:", get_ngrams(tokens, 3))

### Exercise 2 (Easy)

In [ ]:
from collections import Counter

corpus = ["i love nlp", "i love python", "i hate bugs", "nlp is great"]

# Flatten and get bigrams
all_tokens = ' '.join(corpus).split()
bigrams = [(all_tokens[i], all_tokens[i+1]) for i in range(len(all_tokens)-1)]

# Count
bigram_counts = Counter(bigrams)
word_counts = Counter(all_tokens)

# P(love | i)
p_love_given_i = bigram_counts[('i', 'love')] / word_counts['i']
print(f"P(love|i) = {p_love_given_i:.2f}")

### Exercise 3 (Medium)

In [ ]:
from collections import defaultdict, Counter

def build_bigram_model(corpus):
    tokens = corpus.lower().split()
    bigrams = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
    
    model = defaultdict(Counter)
    for w1, w2 in bigrams:
        model[w1][w2] += 1
    
    # Convert to probabilities
    for w1 in model:
        total = sum(model[w1].values())
        for w2 in model[w1]:
            model[w1][w2] /= total
    return model

def predict_next(model, word):
    if word in model:
        return max(model[word], key=model[word].get)
    return None

corpus = "the cat sat on the mat . the dog sat on the rug . the cat is on the mat ."
model = build_bigram_model(corpus)
print(f"After 'the': {predict_next(model, 'the')}")

### Exercise 4 (Medium)

In [ ]:
def build_bigram_model_smoothed(corpus, vocab_size):
    tokens = corpus.lower().split()
    bigrams = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
    
    bigram_counts = Counter(bigrams)
    word_counts = Counter(tokens)
    
    model = defaultdict(dict)
    vocab = set(tokens)
    
    for w1 in vocab:
        for w2 in vocab:
            count = bigram_counts.get((w1, w2), 0)
            model[w1][w2] = (count + 1) / (word_counts[w1] + vocab_size)
    
    return model

corpus = "the cat sat on the mat"
vocab_size = len(set(corpus.split()))
model = build_bigram_model_smoothed(corpus, vocab_size)
print(f"P(cat|the) smoothed: {model['the']['cat']:.4f}")

### Exercise 5 (Hard)

In [ ]:
import math
from collections import Counter, defaultdict

def calculate_perplexity(model, test_sentence):
    tokens = test_sentence.lower().split()
    n = len(tokens)
    log_prob_sum = 0
    
    for i in range(len(tokens)-1):
        w1, w2 = tokens[i], tokens[i+1]
        prob = model.get(w1, {}).get(w2, 1e-10)  # Small prob for unseen
        log_prob_sum += math.log(prob)
    
    perplexity = math.exp(-log_prob_sum / (n-1))
    return perplexity

# Build model and test
corpus = "the cat sat on the mat . the dog sat on the rug ."
model = build_bigram_model_smoothed(corpus, len(set(corpus.split())))

test = "the cat sat on the rug"
print(f"Perplexity: {calculate_perplexity(model, test):.2f}")